# Correctness check — Spark parallelization is exact, not approximate

The paper's §4.2.1 argues that splitting users (then items) across workers changes nothing
mathematically: within one phase, every update reads only *frozen* shared state (`Q` and
`S^q`) and writes just its own `p_u`. Rather than take that on faith, this notebook fits the
same tiny synthetic problem with the sequential numpy oracle (`numpy_reference.py`) and with
Spark (`spark_eals.py`) at three partition counts, and checks the results agree to float
tolerance — the same check `tests/test_spark_eals.py` makes automatically, made visible here.

Runs in a few seconds, no real data needed.

In [1]:
# Input: none. Output: none — puts src/ on the path and starts a local SparkSession.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np

from eals.spark_utils import build_session

spark = build_session("correctness_check", n_cores=4)
sc = spark.sparkContext
print("Spark version:", spark.version, "| master:", sc.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/29 21:58:47 WARN Utils: Your hostname, DESKTOP-APAYAO, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/29 21:58:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/29 21:58:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0 | master: local[4]


## 1. A tiny synthetic problem (no real data needed)

In [2]:
# Input: a random seed. Output: an Interactions object (25 users x 18 items, ~25% density)
# plus every user's items as plain (u, i) pairs, ready to feed both implementations.
from eals import interactions as itx
from eals.weighting import item_confidence


def toy_data(n_users=25, n_items=18, density=0.25, seed=0):
    rng = np.random.default_rng(seed)
    mask = rng.random((n_users, n_items)) < density
    mask[np.arange(n_users), rng.integers(0, n_items, n_users)] = True  # no empty user
    mask[rng.integers(0, n_users, n_items), np.arange(n_items)] = True  # no empty item
    users, items = np.nonzero(mask)
    return itx.build(users, items, n_users, n_items)


data = toy_data()
c = item_confidence(data.item_counts(), c0=32.0, alpha=0.4)
pairs = [
    (int(u), int(i))
    for u in range(data.n_users)
    for i in data.u_items[data.user_slice(u)]
]
print(f"n_users={data.n_users} n_items={data.n_items} nnz={data.nnz}")

n_users=25 n_items=18 nnz=136


## 2. Fit sequentially (numpy) once — the ground truth

In [3]:
# Input: data, c, K=6, 5 iterations, seed=2. Output: the reference (P, Q) every Spark run below is compared against.
from eals import numpy_reference as ref

K, LAM, N_ITERS, SEED = 6, 0.01, 5, 2
expected = ref.fit(data, K=K, c=c, lam=LAM, n_iters=N_ITERS, seed=SEED, track_objective=False)
print("reference P shape:", expected.P.shape, "| Q shape:", expected.Q.shape)

reference P shape: (25, 6) | Q shape: (18, 6)


## 3. Fit in parallel on Spark, at three different partition counts

If the parallelization were only approximately correct, different partition counts would
give different results (more partitions -> more independent "stale-state" workers). If it's
exact, every partition count gives the *same* answer, to float tolerance.

In [4]:
# Input: the same (pairs, K, c, lam, seed) as the numpy fit above, varying only n_parts.
# Output: for each partition count, the max|P_spark - P_numpy| and max|Q_spark - Q_numpy| — should be ~1e-9 or smaller.
from eals import spark_eals

for n_parts in (1, 3, 7):
    user_rdd, item_rdd = spark_eals.build_views(sc, pairs, n_parts=n_parts)
    got = spark_eals.fit(sc, user_rdd, item_rdd, data.n_users, data.n_items, K=K, c=c, lam=LAM, n_iters=N_ITERS, seed=SEED, n_parts=n_parts)

    diff_P = np.max(np.abs(got.P - expected.P))
    diff_Q = np.max(np.abs(got.Q - expected.Q))
    print(f"n_parts={n_parts:2d}  max|dP|={diff_P:.2e}  max|dQ|={diff_Q:.2e}  match={diff_P < 1e-6 and diff_Q < 1e-6}")

n_parts= 1  max|dP|=3.55e-15  max|dQ|=2.78e-16  match=True


n_parts= 3  max|dP|=5.33e-15  max|dQ|=2.50e-16  match=True


n_parts= 7  max|dP|=3.55e-15  max|dQ|=2.22e-16  match=True


## 4. The objective must be non-increasing (coordinate descent's defining property)

Checked on the numpy oracle (cheap to track every iteration there); since §3 already proved
Spark produces the identical `P`/`Q`, this transfers to the Spark result too.

In [5]:
# Input: the same toy data/hyperparameters, this time tracking the objective (Eq. 14) every iteration.
# Output: a printed list of objective values and a boolean confirming they never increase.
model = ref.fit(data, K=K, c=c, lam=LAM, n_iters=10, seed=SEED, track_objective=True)
obj = [h["objective"] for h in model.history]
print("objective per iteration:", [round(o, 3) for o in obj])
print("non-increasing:", all(a >= b - 1e-8 for a, b in zip(obj, obj[1:])))

objective per iteration: [79.469, 60.466, 55.374, 52.491, 51.031, 50.219, 49.67, 49.257, 48.928, 48.659]
non-increasing: True


In [6]:
# Input: none. Output: none — releases the Spark session.
spark.stop()